# ReliableVAEP × 2016 Ballon d'Or — Real Madrid six-player external-agreement check

This standalone Kaggle notebook reproduces the paper's secondary external-agreement analysis.
It trains the canonical VAEP pipeline on the complete **La Liga 2015/16** event sample and
restricts the Ballon d'Or comparison to the six Real Madrid nominees used in the paper:
Cristiano Ronaldo, Gareth Bale, Pepe, Luka Modrić, Toni Kroos, and Sergio Ramos.

The Ballon d'Or is treated as an **external human ranking, not ground truth**. The award covers
a broader calendar-year context than the available domestic-league event data, so the analysis
is intentionally reported as a boundary/external-agreement check.


> **Canonical ReliableVAEP bootstrap setting for the final SSAC analysis:** `FULL_BOOTSTRAP_B = 3500`.
>
> The value is not an arbitrary round number. ReliableVAEP treats a pairwise ordering as supported at
> \(P_{ij}\ge 0.90\) (or \(\le 0.10\)). Requiring a 95% Wilson Monte Carlo half-width of at most
> 0.01 at \(p=0.90\) gives a minimum of 3,461 resamples. The final analyses therefore use 3,500
> player-preserving match-level bootstrap draws, the smallest 100-draw increment above that bound.
> The completed cross-seed sensitivity test confirmed very high rank-order and pair-classification
> stability at \(B=3500\) across the study datasets.
>
> **Important:** `INNER_BOOTSTRAP_B` values used inside repeated split-sample validation are left at
> their original values. They are nested computational approximations for hundreds of repeated
> validation runs and are not used to produce the paper's primary full-data rank intervals or
> 0.90 pairwise-support decisions.

In [1]:
# FINAL SSAC RUN CONFIGURATION CHECK
CANONICAL_FULL_BOOTSTRAP_B = 3500
print("ReliableVAEP final SSAC run")
print("Canonical full-data bootstrap draws:", CANONICAL_FULL_BOOTSTRAP_B)
print("Run this notebook from top to bottom and use only the newly generated outputs.")


ReliableVAEP final SSAC run
Canonical full-data bootstrap draws: 3500
Run this notebook from top to bottom and use only the newly generated outputs.


In [2]:

# ================================================================
# CELL 1 — KAGGLE-SAFE ISOLATED ENVIRONMENT
# ================================================================
#
# IMPORTANT:
# Kaggle Internet must be ON.
#
# This setup:
#   1. uses /kaggle/working only;
#   2. installs only the small pure-Python "virtualenv" helper into
#      Kaggle's host environment if it is missing;
#   3. creates /kaggle/working/rv_vaep_env;
#   4. installs the complete VAEP stack INSIDE that environment;
#   5. leaves Kaggle's NumPy/pandas/scipy/sklearn untouched.
#
# No runtime restart is required.
# ================================================================

import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

if not Path("/kaggle/working").exists():
    raise RuntimeError(
        "This notebook is the Kaggle-specific version, but /kaggle/working "
        "was not found. Please run it inside a Kaggle Notebook."
    )

if sys.version_info >= (3, 13):
    raise RuntimeError(
        "Please use a Kaggle Python runtime <= 3.12. "
        "socceraction 1.5.3 is not designed for Python 3.13."
    )

BASE_DIR = Path("/kaggle/working")
ENV_DIR = BASE_DIR / "rv_vaep_env"
ENV_PY = ENV_DIR / "bin" / "python"
ENV_PIP = ENV_DIR / "bin" / "pip"
READY = ENV_DIR / ".reliablevaep_ballondor_laliga_ready_v1"

IMPORT_TEST = r"""
import numpy
import pandas
import scipy
import sklearn
import xgboost
import socceraction

from socceraction.data.statsbomb import StatsBombLoader
from socceraction.vaep import VAEP
import socceraction.spadl as spadl

print("numpy        =", numpy.__version__)
print("pandas       =", pandas.__version__)
print("scipy        =", scipy.__version__)
print("sklearn      =", sklearn.__version__)
print("xgboost      =", xgboost.__version__)
print("socceraction =", getattr(socceraction, "__version__", "installed"))

# NumPy/pandas binary compatibility smoke test.
df = pandas.DataFrame({"x": numpy.arange(5, dtype=float)})
assert float(df["x"].sum()) == 10.0

print("KAGGLE_ISOLATED_ENV_OK")
"""


def run_capture(args, timeout=None):
    clean_env = os.environ.copy()
    clean_env.pop("PYTHONPATH", None)
    clean_env["PYTHONNOUSERSITE"] = "1"
    return subprocess.run(
        [str(x) for x in args],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
        env=clean_env,
    )


def environment_is_healthy():
    if not (
        ENV_PY.exists()
        and ENV_PIP.exists()
        and READY.exists()
    ):
        return False

    test = run_capture(
        [ENV_PY, "-c", IMPORT_TEST]
    )

    if (
        test.returncode == 0
        and "KAGGLE_ISOLATED_ENV_OK" in test.stdout
    ):
        print("Existing isolated environment verified:")
        print(test.stdout)
        return True

    print("Existing isolated environment is incomplete/unhealthy.")
    print(test.stdout)
    return False


if not environment_is_healthy():

    # Remove only our own previous environment.
    if ENV_DIR.exists():
        print("Removing incomplete ReliableVAEP environment:", ENV_DIR)
        shutil.rmtree(ENV_DIR, ignore_errors=True)

    # ------------------------------------------------------------
    # Install virtualenv only if unavailable.
    # virtualenv is pure Python and does not alter NumPy/pandas.
    # ------------------------------------------------------------
    if importlib.util.find_spec("virtualenv") is None:
        print("Installing the lightweight virtualenv helper into Kaggle...")
        helper = run_capture([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-cache-dir",
            "virtualenv==20.28.0",
        ])

        print(helper.stdout)

        if helper.returncode != 0:
            raise RuntimeError(
                "Could not install virtualenv.\n"
                "Make sure Kaggle Internet is ON.\n\n"
                + helper.stdout
            )
    else:
        print("virtualenv helper is already available.")

    # ------------------------------------------------------------
    # Create isolated environment using virtualenv.
    # This avoids system ensurepip entirely.
    # ------------------------------------------------------------
    print("\nCreating isolated environment:", ENV_DIR)

    create = run_capture([
        sys.executable,
        "-m",
        "virtualenv",
        "--clear",
        str(ENV_DIR),
    ])

    print(create.stdout)

    if (
        create.returncode != 0
        or not ENV_PY.exists()
        or not ENV_PIP.exists()
    ):
        raise RuntimeError(
            "virtualenv could not create the isolated environment.\n\n"
            + create.stdout
        )

    # ------------------------------------------------------------
    # Install one internally consistent stack.
    # These packages exist ONLY inside /kaggle/working/rv_vaep_env.
    # ------------------------------------------------------------
    packages = [
        "numpy==1.26.4",
        "pandas==2.1.4",
        "scipy==1.13.1",
        "scikit-learn==1.5.2",
        "xgboost==2.1.4",
        "matplotlib==3.9.2",
        "tqdm==4.66.5",
        "multimethod==1.10",
        "pandera==0.17.2",
        "socceraction[statsbomb]==1.5.3",
    ]

    print("\nInstalling the isolated ReliableVAEP stack...")
    print("This can take a few minutes on the first run.")

    install = run_capture([
        ENV_PY,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        *packages,
    ])

    print(install.stdout)

    if install.returncode != 0:
        raise RuntimeError(
            "Package installation failed INSIDE the isolated environment.\n"
            "Kaggle's host NumPy/pandas environment has not been changed.\n\n"
            + install.stdout
        )

    # ------------------------------------------------------------
    # Verify dependency consistency.
    # ------------------------------------------------------------
    print("\nChecking isolated dependency consistency...")

    check = run_capture([
        ENV_PY,
        "-m",
        "pip",
        "check",
    ])

    print(
        check.stdout
        if check.stdout.strip()
        else "pip check: no dependency conflicts"
    )

    if check.returncode != 0:
        raise RuntimeError(
            "Dependency conflicts remain inside the isolated environment.\n\n"
            + check.stdout
        )

    # ------------------------------------------------------------
    # Import and binary compatibility test.
    # ------------------------------------------------------------
    print("\nVerifying isolated imports...")

    verify = run_capture([
        ENV_PY,
        "-c",
        IMPORT_TEST,
    ])

    print(verify.stdout)

    if (
        verify.returncode != 0
        or "KAGGLE_ISOLATED_ENV_OK" not in verify.stdout
    ):
        raise RuntimeError(
            "The isolated environment was created, but its scientific stack "
            "failed verification.\n\n"
            + verify.stdout
        )

    READY.write_text("ready", encoding="utf-8")

    print("\nKaggle isolated environment setup completed successfully.")

else:
    print("\nReusing the verified Kaggle isolated environment.")

print("\nAnalysis Python:", ENV_PY)
print("Kaggle host scientific packages were NOT downgraded or replaced.")


Installing the lightweight virtualenv helper into Kaggle...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 341.2 MB/s eta 0:00:00


Creating isolated environment: /kaggle/working/rv_vaep_env
created virtual environment CPython3.12.13.final.0-64 in 334ms
  creator CPython3Posix(dest=/kaggle/working/rv_vaep_env, clear=True, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.local/share/virtualenv)
    added seed packages: pip==24.3.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator


Installing the isolated ReliableVAEP stack...
This can take a few minutes on the first run.
Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
INFO: pip is looking at multiple versions of contourpy to determine which version is compatible with other 

## Analysis script

The next cell writes the complete standalone analysis. No previous ReliableVAEP files are required in Kaggle.


In [3]:
# ================================================================
# CELL 3 — WRITE THE COMPLETE STANDALONE ANALYSIS SCRIPT
# ================================================================
from pathlib import Path

SCRIPT_PATH = Path("/kaggle/working/reliablevaep_ballondor2016_laliga.py")

ANALYSIS_SCRIPT = '\n# ================================================================\n# ReliableVAEP — 2016 Ballon d\'Or × Real Madrid six-player external-agreement check\n# FULLY STANDALONE KAGGLE ANALYSIS\n# ================================================================\n\nimport json\nimport os\nimport shutil\nimport time\nimport urllib.request\nimport warnings\nimport unicodedata\nimport re\nfrom pathlib import Path\n\n# Kaggle exports a notebook-only matplotlib backend.  The analysis\n# runs in an isolated subprocess, so force a normal file backend first.\nos.environ["MPLBACKEND"] = "Agg"\n\nimport matplotlib\nmatplotlib.use("Agg", force=True)\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport xgboost as xgb\n\nfrom scipy.optimize import minimize\nfrom scipy.special import expit, logit\nfrom scipy.stats import spearmanr, kendalltau, wilcoxon\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score\nfrom sklearn.model_selection import GroupKFold, GroupShuffleSplit\nfrom tqdm.auto import tqdm\n\nfrom socceraction.data.statsbomb import StatsBombLoader\nimport socceraction.spadl as spadl\nfrom socceraction.vaep import VAEP\nfrom socceraction.vaep import formula as vaep_formula\n\nwarnings.filterwarnings("ignore")\n\n# ----------------------------------------------------------------\n# Reproducibility and analysis settings\n# ----------------------------------------------------------------\nSEED = 20260919\nnp.random.seed(SEED)\n\n# Primary external-validity design:\n# 2016 Ballon d\'Or nominees who played in La Liga 2015/16.\nCOMPETITION_ID = 11\nSEASON_ID = 27\nDATASET_LABEL = "La Liga 2015/16 — Real Madrid 2016 Ballon d\'Or six-player cohort"\nEXPECTED_LEAGUE_MATCHES = 380\nMIN_SOURCE_COVERAGE = 0.95\n\n# Same base eligibility rule used in ReliableVAEP.\nMIN_MATCHES = 5\nMIN_MINUTES = 300\nMIN_ACTIONS = 150\n\n# Full static ReliableVAEP / PCS / RAR analysis.\nFULL_BOOTSTRAP_B = 3500\nPAIRWISE_SUPPORT = 0.90\nBT_RIDGE = 1e-4\n\n# External-agreement stress test:\n# draw n domestic-league matches per nominee, rebuild raw VAEP and RAR,\n# then compare each ranking with the independent Ballon d\'Or ordering.\nEXTERNAL_N_VALUES = [5, 8, 10]\nEXTERNAL_REPS = 200\nINNER_BOOTSTRAP_B = 500\n\nOUTER_FOLDS = 5\nCALIBRATION_MATCH_FRACTION = 0.20\n\nROOT = Path("/kaggle/working")\nCACHE_ROOT = ROOT / "rv_ballondor2016_laliga_cache"\nOUT_ROOT = ROOT / "rv_ballondor2016_laliga_outputs"\n\nCACHE_ROOT.mkdir(parents=True, exist_ok=True)\nOUT_ROOT.mkdir(parents=True, exist_ok=True)\n\nBASE_URL = "https://raw.githubusercontent.com/hudl/open-data/master/data"\n\n# ----------------------------------------------------------------\n# Official 2016 Ballon d\'Or results for the La Liga 2015/16 nominees.\n#\n# Published ranking/points:\n# L\'Équipe, 12 Dec 2016:\n# https://www.lequipe.fr/Football/Actualites/Le-classement-du-ballon-d-or-france-football-2016-eg/759441\n#\n# UEFA independently confirms the top ten:\n# https://www.uefa.com/uefachampionsleague/news/0234-0e95e3ad8e7e-790316fff6fb-1000--cristiano-ronaldo-wins-ballon-dor/\n#\n# The Ballon d\'Or is an external human-voting reference, NOT ground truth.\n# It covers the 2016 award cycle, while the event data below are domestic\n# league matches from 2015/16. That temporal/scope mismatch is reported\n# explicitly as a limitation.\n# ----------------------------------------------------------------\n\nBALLON_DOR = [\n    {"ballon_name": "Cristiano Ronaldo", "published_rank": 1, "points": 745, "team_2015_16": "Real Madrid", "aliases": [["cristiano", "ronaldo"]]},\n    {"ballon_name": "Gareth Bale", "published_rank": 6, "points": 60, "team_2015_16": "Real Madrid", "aliases": [["gareth", "bale"]]},\n    {"ballon_name": "Pepe", "published_rank": 9, "points": 8, "team_2015_16": "Real Madrid", "aliases": [["pepe"], ["kepler", "laveran"], ["laveran", "ferreira"], ["lima", "ferreira"]]},\n    {"ballon_name": "Luka Modrić", "published_rank": 17, "points": 1, "team_2015_16": "Real Madrid", "aliases": [["luka", "modric"]]},\n    {"ballon_name": "Toni Kroos", "published_rank": 17, "points": 1, "team_2015_16": "Real Madrid", "aliases": [["toni", "kroos"]]},\n    {"ballon_name": "Sergio Ramos", "published_rank": 20, "points": 0, "team_2015_16": "Real Madrid", "aliases": [["sergio", "ramos"]]},\n]\n\nprint("=" * 96)\nprint("ReliableVAEP external validity: 2016 Ballon d\'Or × La Liga 2015/16")\nprint("=" * 96)\nprint("Primary comparison cohort:", len(BALLON_DOR), "Real Madrid Ballon d\'Or nominees")\nprint("Base eligibility:",\n      f">={MIN_MATCHES} matches, >={MIN_MINUTES} minutes, >={MIN_ACTIONS} actions")\nprint("Full bootstrap draws:", FULL_BOOTSTRAP_B)\nprint("Pairwise support threshold:", PAIRWISE_SUPPORT)\nprint("External-agreement n values:", EXTERNAL_N_VALUES)\nprint("External repetitions per n:", EXTERNAL_REPS)\nprint("Inner bootstrap draws:", INNER_BOOTSTRAP_B)\nprint()\nprint("IMPORTANT: Ballon d\'Or is treated as an independent external ranking, not ground truth.")\nprint("IMPORTANT: 2016 award scope and 2015/16 domestic-league scope are not identical.")\nprint()\n\n# ================================================================\n# Utility functions\n# ================================================================\n\ndef get_json(url, path, retries=5):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n\n    if path.exists():\n        try:\n            with open(path, encoding="utf-8") as f:\n                return json.load(f)\n        except Exception:\n            path.unlink(missing_ok=True)\n\n    for attempt in range(1, retries + 1):\n        try:\n            req = urllib.request.Request(\n                url,\n                headers={"User-Agent": "ReliableVAEP-BallonDor2016-LaLiga/1.0"},\n            )\n            with urllib.request.urlopen(req, timeout=120) as response:\n                obj = json.loads(response.read().decode("utf-8"))\n            with open(path, "w", encoding="utf-8") as f:\n                json.dump(obj, f)\n            return obj\n        except Exception as exc:\n            if attempt == retries:\n                raise RuntimeError(f"Download failed: {url}") from exc\n            time.sleep(min(2 ** attempt, 15))\n\n\ndef make_model(seed):\n    return xgb.XGBClassifier(\n        n_estimators=180,\n        max_depth=3,\n        learning_rate=0.05,\n        min_child_weight=5,\n        subsample=0.85,\n        colsample_bytree=0.85,\n        reg_lambda=2.0,\n        objective="binary:logistic",\n        eval_metric="logloss",\n        tree_method="hist",\n        random_state=seed,\n        n_jobs=-1,\n    )\n\n\ndef fit_platt(prob, y):\n    prob = np.asarray(prob, dtype=float)\n    y = np.asarray(y, dtype=int)\n    if len(np.unique(y)) < 2:\n        return None\n    z = logit(np.clip(prob, 1e-6, 1 - 1e-6)).reshape(-1, 1)\n    model = LogisticRegression(solver="lbfgs", max_iter=1000)\n    model.fit(z, y)\n    return model\n\n\ndef apply_platt(model, prob):\n    prob = np.asarray(prob, dtype=float)\n    if model is None:\n        return prob.copy()\n    z = logit(np.clip(prob, 1e-6, 1 - 1e-6)).reshape(-1, 1)\n    return model.predict_proba(z)[:, 1]\n\n\ndef safe_auc(y, p):\n    return float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan\n\n\ndef baseline_brier(y):\n    y = np.asarray(y, dtype=float)\n    return float(np.mean((y - y.mean()) ** 2))\n\n\ndef vaep90(q):\n    minutes = float(q["minutes_played"].sum())\n    if minutes <= 0:\n        return np.nan\n    return float(90.0 * q["vaep"].sum() / minutes)\n\n\ndef ranks_desc(values):\n    return (\n        pd.Series(np.asarray(values, dtype=float))\n        .rank(ascending=False, method="average")\n        .to_numpy(dtype=float)\n    )\n\n\ndef player_id_to_jsonable(x):\n    if isinstance(x, (np.integer, int)):\n        return int(x)\n    try:\n        xf = float(x)\n        if xf.is_integer():\n            return int(xf)\n        return xf\n    except Exception:\n        return str(x)\n\n\nSBL = StatsBombLoader(getter="remote", creds=None)\n\n\n# ================================================================\n# STEP 1 — Canonical La Liga 2015/16 player-match VAEP\n# ================================================================\n\ndef build_laliga_player_match():\n    dataset_dir = OUT_ROOT / "canonical_build"\n    dataset_dir.mkdir(parents=True, exist_ok=True)\n\n    marker = dataset_dir / "BUILD_COMPLETE.json"\n    pm_pickle = dataset_dir / "player_match_vaep.pkl"\n    cohort_csv = dataset_dir / "eligible_cohort.csv"\n    metrics_csv = dataset_dir / "crossfit_model_metrics.csv"\n\n    if marker.exists() and pm_pickle.exists() and cohort_csv.exists() and metrics_csv.exists():\n        try:\n            meta = json.loads(marker.read_text(encoding="utf-8"))\n            pm = pd.read_pickle(pm_pickle)\n            cohort = pd.read_csv(cohort_csv)\n            metrics = pd.read_csv(metrics_csv)\n            print("Loading completed cached canonical La Liga build.")\n            print("Matches:", meta["matches"])\n            print("Eligible players:", meta["eligible_players"])\n            return pm, cohort, metrics, meta\n        except Exception as exc:\n            print("Cached build could not be loaded; rebuilding:", exc)\n\n    # ------------------------------------------------------------\n    # Match list\n    # ------------------------------------------------------------\n    match_file = CACHE_ROOT / "matches" / f"{COMPETITION_ID}_{SEASON_ID}.json"\n    matches_raw = get_json(\n        f"{BASE_URL}/matches/{COMPETITION_ID}/{SEASON_ID}.json",\n        match_file,\n    )\n\n    games = pd.DataFrame([\n        {\n            "game_id": int(m["match_id"]),\n            "game_date": pd.to_datetime(m["match_date"]),\n            "home_team_id": int(m["home_team"]["home_team_id"]),\n            "away_team_id": int(m["away_team"]["away_team_id"]),\n            "home_team_name": str(m["home_team"]["home_team_name"]),\n            "away_team_name": str(m["away_team"]["away_team_name"]),\n        }\n        for m in matches_raw\n    ]).sort_values(["game_date", "game_id"]).reset_index(drop=True)\n\n    observed_matches = int(games["game_id"].nunique())\n    source_coverage = observed_matches / EXPECTED_LEAGUE_MATCHES\n\n    print(\n        "StatsBomb La Liga 2015/16 matches found:",\n        observed_matches,\n        f"({100*source_coverage:.1f}% of {EXPECTED_LEAGUE_MATCHES})",\n    )\n\n    if source_coverage < MIN_SOURCE_COVERAGE:\n        raise RuntimeError(\n            f"Insufficient La Liga source coverage: {observed_matches}/"\n            f"{EXPECTED_LEAGUE_MATCHES} ({100*source_coverage:.1f}%), below "\n            f"the declared {100*MIN_SOURCE_COVERAGE:.1f}% minimum."\n        )\n\n    games.to_csv(dataset_dir / "game_list.csv", index=False)\n\n    # ------------------------------------------------------------\n    # SPADL actions + official player minutes\n    # ------------------------------------------------------------\n    store = {}\n\n    for g in tqdm(\n        games.itertuples(index=False),\n        total=len(games),\n        desc="La Liga 2015/16: SPADL",\n        unit="match",\n    ):\n        gid = int(g.game_id)\n\n        action_path = CACHE_ROOT / "spadl" / f"{gid}.pkl"\n        player_path = CACHE_ROOT / "players" / f"{gid}.pkl"\n        action_path.parent.mkdir(parents=True, exist_ok=True)\n        player_path.parent.mkdir(parents=True, exist_ok=True)\n\n        if action_path.exists():\n            actions = pd.read_pickle(action_path)\n        else:\n            events = SBL.events(game_id=gid)\n            actions = (\n                spadl.statsbomb.convert_to_actions(\n                    events=events,\n                    home_team_id=int(g.home_team_id),\n                )\n                .reset_index(drop=True)\n            )\n            actions.to_pickle(action_path)\n\n        if player_path.exists():\n            players = pd.read_pickle(player_path)\n        else:\n            players = SBL.players(game_id=gid)\n            players.to_pickle(player_path)\n\n        if len(actions) == 0:\n            raise RuntimeError(f"No SPADL actions in match {gid}.")\n\n        store[gid] = {\n            "game": pd.Series({\n                "game_id": gid,\n                "home_team_id": int(g.home_team_id),\n                "away_team_id": int(g.away_team_id),\n                "home_team_name": str(g.home_team_name),\n                "away_team_name": str(g.away_team_name),\n                "game_date": g.game_date,\n            }),\n            "actions": actions,\n            "players": players,\n        }\n\n    # ------------------------------------------------------------\n    # Canonical VAEP features and labels\n    # ------------------------------------------------------------\n    template = VAEP()\n\n    X_parts = []\n    Y_parts = []\n    row_parts = []\n\n    for gid, item in tqdm(store.items(), desc="Canonical VAEP features", unit="match"):\n        Xg = template.compute_features(\n            item["game"],\n            item["actions"],\n        ).reset_index(drop=True)\n\n        Yg = template.compute_labels(\n            item["game"],\n            item["actions"],\n        ).reset_index(drop=True)\n\n        if len(Xg) != len(item["actions"]) or len(Yg) != len(item["actions"]):\n            raise RuntimeError(f"Feature/label/action alignment failed in match {gid}.")\n\n        X_parts.append(Xg)\n        Y_parts.append(Yg)\n        row_parts.append(\n            pd.DataFrame({\n                "game_id": gid,\n                "action_row": np.arange(len(Xg)),\n            })\n        )\n\n    X = pd.concat(X_parts, ignore_index=True)\n    Y = pd.concat(Y_parts, ignore_index=True)\n    ROW = pd.concat(row_parts, ignore_index=True)\n\n    # Convert any nonnumeric encoded feature columns to numeric forms.\n    for col in X.columns:\n        if pd.api.types.is_bool_dtype(X[col]):\n            X[col] = X[col].astype(np.uint8)\n        elif isinstance(X[col].dtype, pd.CategoricalDtype):\n            X[col] = X[col].cat.codes.astype(np.int32)\n        elif pd.api.types.is_object_dtype(X[col]):\n            X[col] = pd.factorize(X[col].astype(str))[0].astype(np.int32)\n\n    bad_features = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]\n    if bad_features:\n        raise TypeError(f"Non-numeric VAEP feature columns remain: {bad_features}")\n\n    print(\n        f"Action rows={len(X):,}, features={X.shape[1]}, "\n        f"score rate={Y[\'scores\'].mean():.4f}, concede rate={Y[\'concedes\'].mean():.4f}"\n    )\n\n    # ------------------------------------------------------------\n    # Match-level out-of-fold prediction + inner calibration\n    # ------------------------------------------------------------\n    groups = ROW["game_id"].to_numpy()\n    unique_matches = np.unique(groups)\n    n_folds = min(OUTER_FOLDS, len(unique_matches))\n\n    outer = GroupKFold(n_splits=n_folds)\n\n    oof = {\n        "scores_raw": np.full(len(X), np.nan),\n        "scores_cal": np.full(len(X), np.nan),\n        "concedes_raw": np.full(len(X), np.nan),\n        "concedes_cal": np.full(len(X), np.nan),\n    }\n\n    metric_rows = []\n\n    for fold, (train_idx, test_idx) in enumerate(\n        outer.split(X, groups=groups),\n        start=1,\n    ):\n        inner = GroupShuffleSplit(\n            n_splits=1,\n            test_size=CALIBRATION_MATCH_FRACTION,\n            random_state=SEED + fold,\n        )\n\n        fit_rel, cal_rel = next(\n            inner.split(\n                X.iloc[train_idx],\n                groups=groups[train_idx],\n            )\n        )\n\n        fit_idx = train_idx[fit_rel]\n        cal_idx = train_idx[cal_rel]\n\n        row = {\n            "fold": fold,\n            "fit_matches": len(np.unique(groups[fit_idx])),\n            "calibration_matches": len(np.unique(groups[cal_idx])),\n            "test_matches": len(np.unique(groups[test_idx])),\n        }\n\n        for target in ["scores", "concedes"]:\n            model = make_model(\n                SEED + fold * 10 + (1 if target == "scores" else 2)\n            )\n\n            model.fit(\n                X.iloc[fit_idx],\n                Y.iloc[fit_idx][target].astype(int),\n            )\n\n            p_cal_raw = model.predict_proba(X.iloc[cal_idx])[:, 1]\n            calibrator = fit_platt(\n                p_cal_raw,\n                Y.iloc[cal_idx][target].astype(int),\n            )\n\n            p_test_raw = model.predict_proba(X.iloc[test_idx])[:, 1]\n            p_test_cal = apply_platt(calibrator, p_test_raw)\n\n            oof[f"{target}_raw"][test_idx] = p_test_raw\n            oof[f"{target}_cal"][test_idx] = p_test_cal\n\n            y_test = Y.iloc[test_idx][target].astype(int).to_numpy()\n            base = baseline_brier(y_test)\n\n            row[f"{target}_auc"] = safe_auc(y_test, p_test_cal)\n            row[f"{target}_ap"] = float(average_precision_score(y_test, p_test_cal))\n            row[f"{target}_brier_baseline"] = base\n            row[f"{target}_brier_raw"] = float(brier_score_loss(y_test, p_test_raw))\n            row[f"{target}_brier_cal"] = float(brier_score_loss(y_test, p_test_cal))\n            row[f"{target}_brier_skill_cal"] = (\n                1.0 - row[f"{target}_brier_cal"] / base if base > 0 else np.nan\n            )\n\n        metric_rows.append(row)\n\n    for key, values in oof.items():\n        if np.isnan(values).any():\n            raise RuntimeError(f"Incomplete OOF predictions: {key}")\n\n    metrics = pd.DataFrame(metric_rows)\n    metrics.to_csv(metrics_csv, index=False)\n\n    print("\\nCross-fit model means:")\n    print(\n        metrics.mean(numeric_only=True)\n        .round(4)\n        .to_string()\n    )\n\n    # ------------------------------------------------------------\n    # Canonical action VAEP\n    # ------------------------------------------------------------\n    scored_parts = []\n    cursor = 0\n\n    for gid, item in tqdm(store.items(), desc="Canonical action VAEP", unit="match"):\n        actions = item["actions"].reset_index(drop=True)\n        n_actions = len(actions)\n\n        ps = pd.Series(oof["scores_cal"][cursor:cursor + n_actions])\n        pc = pd.Series(oof["concedes_cal"][cursor:cursor + n_actions])\n\n        named = spadl.add_names(actions.copy()).reset_index(drop=True)\n        values = vaep_formula.value(named, ps, pc).reset_index(drop=True)\n\n        z = named.copy()\n        z["game_id"] = gid\n        z["game_date"] = item["game"]["game_date"]\n        z["p_score"] = ps.to_numpy()\n        z["p_concede"] = pc.to_numpy()\n\n        for col in ["offensive_value", "defensive_value", "vaep_value"]:\n            z[col] = values[col].to_numpy()\n\n        scored_parts.append(z)\n        cursor += n_actions\n\n    if cursor != len(X):\n        raise RuntimeError("OOF probability/action alignment failed.")\n\n    scored = pd.concat(scored_parts, ignore_index=True)\n\n    # ------------------------------------------------------------\n    # Goal sanity check\n    # ------------------------------------------------------------\n    goal_mask = (\n        scored["type_name"].isin(["shot", "shot_freekick", "shot_penalty"])\n        & scored["result_name"].eq("success")\n    )\n\n    if int(goal_mask.sum()) == 0:\n        raise RuntimeError("No successful goals detected for VAEP sanity checking.")\n\n    goal_values = scored.loc[goal_mask, "vaep_value"]\n    median_goal_vaep = float(goal_values.median())\n    positive_goal_fraction = float((goal_values > 0).mean())\n\n    print(\n        "\\nGoal-value sanity check:",\n        f"n={int(goal_mask.sum())}, "\n        f"median successful-goal VAEP={median_goal_vaep:.4f}, "\n        f"positive fraction={positive_goal_fraction:.3f}"\n    )\n\n    if median_goal_vaep <= 0:\n        raise RuntimeError(\n            "Canonical VAEP sanity check failed: median successful-goal value is not positive."\n        )\n\n    # ------------------------------------------------------------\n    # Player-match aggregation for ALL teams\n    # ------------------------------------------------------------\n    pm_parts = []\n\n    keeper_types = {\n        "keeper_save",\n        "keeper_claim",\n        "keeper_punch",\n        "keeper_pick_up",\n    }\n\n    keeper_action_ids = set(\n        scored.loc[\n            scored["type_name"].isin(keeper_types),\n            "player_id",\n        ]\n        .dropna()\n        .tolist()\n    )\n\n    for gid, item in tqdm(store.items(), desc="Player-match VAEP", unit="match"):\n        players = item["players"].copy()\n        players = players[\n            players["minutes_played"].fillna(0) > 0\n        ].copy()\n\n        ga = scored[scored["game_id"] == gid].copy()\n\n        agg = (\n            ga.dropna(subset=["player_id"])\n            .groupby("player_id", as_index=False)\n            .agg(\n                vaep=("vaep_value", "sum"),\n                offensive_value=("offensive_value", "sum"),\n                defensive_value=("defensive_value", "sum"),\n                actions=("action_id", "size"),\n            )\n        )\n\n        goals = (\n            ga[\n                ga["type_name"].isin(["shot", "shot_freekick", "shot_penalty"])\n                & ga["result_name"].eq("success")\n            ]\n            .dropna(subset=["player_id"])\n            .groupby("player_id")\n            .size()\n            .rename("goals")\n            .reset_index()\n        )\n\n        agg = agg.merge(goals, on="player_id", how="left")\n        agg["goals"] = agg["goals"].fillna(0).astype(int)\n\n        q = players.merge(agg, on="player_id", how="left")\n\n        for col in ["vaep", "offensive_value", "defensive_value"]:\n            q[col] = q[col].fillna(0.0)\n\n        for col in ["actions", "goals"]:\n            q[col] = q[col].fillna(0).astype(int)\n\n        q["game_id"] = gid\n        q["game_date"] = item["game"]["game_date"]\n\n        team_name_map = {\n            int(item["game"]["home_team_id"]): str(item["game"]["home_team_name"]),\n            int(item["game"]["away_team_id"]): str(item["game"]["away_team_name"]),\n        }\n        if "team_id" in q.columns:\n            q["team_name"] = q["team_id"].map(team_name_map)\n        else:\n            q["team_name"] = ""\n\n        q["vaep90_match"] = (\n            90.0 * q["vaep"] / q["minutes_played"]\n        )\n\n        if "starting_position_name" in q.columns:\n            q["is_goalkeeper_position"] = (\n                q["starting_position_name"]\n                .astype(str)\n                .str.contains("Goalkeeper", case=False, na=False)\n            )\n        else:\n            q["is_goalkeeper_position"] = False\n\n        q["is_goalkeeper"] = (\n            q["is_goalkeeper_position"].astype(bool)\n            | q["player_id"].isin(keeper_action_ids)\n        )\n\n        pm_parts.append(q)\n\n    pm = pd.concat(pm_parts, ignore_index=True)\n\n    # ------------------------------------------------------------\n    # Eligibility\n    # ------------------------------------------------------------\n    cohort = (\n        pm.groupby(["player_id", "player_name"], as_index=False)\n        .agg(\n            total_matches=("game_id", "nunique"),\n            total_minutes=("minutes_played", "sum"),\n            total_actions=("actions", "sum"),\n            total_vaep=("vaep", "sum"),\n            offensive_value=("offensive_value", "sum"),\n            defensive_value=("defensive_value", "sum"),\n            goals=("goals", "sum"),\n            goalkeeper_match_rows=("is_goalkeeper", "sum"),\n            team_name=("team_name", lambda s: "|".join(sorted({\n                str(x) for x in s.dropna().astype(str) if str(x)\n            }))),\n        )\n    )\n\n    cohort["vaep90"] = (\n        90.0 * cohort["total_vaep"] / cohort["total_minutes"]\n    )\n\n    cohort["is_goalkeeper"] = cohort["goalkeeper_match_rows"] > 0\n\n    cohort = cohort[\n        (cohort["total_matches"] >= MIN_MATCHES)\n        & (cohort["total_minutes"] >= MIN_MINUTES)\n        & (cohort["total_actions"] >= MIN_ACTIONS)\n    ].copy()\n\n    cohort = (\n        cohort.sort_values("vaep90", ascending=False)\n        .reset_index(drop=True)\n    )\n\n    cohort["raw_VAEP_rank"] = np.arange(1, len(cohort) + 1)\n\n    print("\\nEligible players:", len(cohort))\n    print("Goalkeepers among eligible players:", int(cohort["is_goalkeeper"].sum()))\n\n    if len(cohort) < 200:\n        print(\n            "WARNING: fewer than 200 base-eligible La Liga players were produced. "\n            "Inspect source coverage and eligibility outputs before interpretation."\n        )\n\n    pm.to_pickle(pm_pickle)\n    pm.to_csv(dataset_dir / "player_match_vaep.csv", index=False)\n    cohort.to_csv(cohort_csv, index=False)\n\n    meta = {\n        "competition_id": COMPETITION_ID,\n        "season_id": SEASON_ID,\n        "expected_matches": int(EXPECTED_LEAGUE_MATCHES),\n        "matches": int(games["game_id"].nunique()),\n        "source_coverage_fraction": float(source_coverage),\n        "eligible_players": int(len(cohort)),\n        "eligible_goalkeepers": int(cohort["is_goalkeeper"].sum()),\n        "median_successful_goal_vaep": median_goal_vaep,\n        "positive_goal_fraction": positive_goal_fraction,\n    }\n\n    marker.write_text(json.dumps(meta, indent=2), encoding="utf-8")\n\n    return pm, cohort, metrics, meta\n\n\n# ================================================================\n# STEP 2 — Full-data match bootstrap\n# ================================================================\n\ndef bootstrap_full_cohort(pm, cohort, B=FULL_BOOTSTRAP_B, seed=SEED + 100):\n    player_ids = cohort["player_id"].tolist()\n    K = len(player_ids)\n\n    grouped = {\n        pid: q.reset_index(drop=True)\n        for pid, q in pm[pm["player_id"].isin(player_ids)].groupby("player_id")\n    }\n\n    V = np.empty((B, K), dtype=float)\n\n    rng = np.random.default_rng(seed)\n\n    for j, pid in enumerate(\n        tqdm(player_ids, desc="Full-cohort match bootstrap", unit="player")\n    ):\n        q = grouped[pid]\n        va = q["vaep"].to_numpy(dtype=float)\n        mi = q["minutes_played"].to_numpy(dtype=float)\n        m = len(q)\n\n        idx = rng.integers(0, m, size=(B, m))\n        total_v = va[idx].sum(axis=1)\n        total_m = mi[idx].sum(axis=1)\n\n        V[:, j] = (\n            90.0 * total_v / np.where(total_m > 0, total_m, np.nan)\n        )\n\n    if not np.isfinite(V).all():\n        raise RuntimeError("Non-finite bootstrap VAEP/90 values detected.")\n\n    # Rank every bootstrap leaderboard across all eligible players.\n    rank_draws = (\n        pd.DataFrame(V)\n        .rank(axis=1, ascending=False, method="average")\n        .to_numpy(dtype=float)\n    )\n\n    return player_ids, V, rank_draws\n\n\ndef pairwise_probability_matrix(V):\n    V = np.asarray(V, dtype=float)\n    K = V.shape[1]\n    P = np.full((K, K), 0.5, dtype=float)\n\n    for i in tqdm(range(K), desc="Pairwise probabilities", unit="player"):\n        for j in range(i + 1, K):\n            d = V[:, i] - V[:, j]\n            p = float(\n                np.mean(\n                    np.where(\n                        d > 0,\n                        1.0,\n                        np.where(d == 0, 0.5, 0.0),\n                    )\n                )\n            )\n            P[i, j] = p\n            P[j, i] = 1.0 - p\n\n    return P\n\n\n# ================================================================\n# STEP 3 — PCS / Bradley-Terry consensus\n# ================================================================\n\ndef fit_pairwise_consensus(P, ridge=BT_RIDGE):\n    P = np.asarray(P, dtype=float)\n    K = P.shape[0]\n\n    if K == 1:\n        return np.array([0.0]), {\n            "success": True,\n            "pairwise_mae": 0.0,\n            "pairwise_rmse": 0.0,\n            "objective": 0.0,\n        }\n\n    ii, jj = np.triu_indices(K, k=1)\n    target = P[ii, jj]\n    valid = np.isfinite(target)\n    ii, jj, target = ii[valid], jj[valid], target[valid]\n\n    def loss_grad(s):\n        z = s[ii] - s[jj]\n        losses = np.logaddexp(0.0, z) - target * z\n        loss = float(np.mean(losses) + ridge * np.mean(s ** 2))\n\n        err = (expit(z) - target) / len(target)\n        grad = np.zeros_like(s)\n        np.add.at(grad, ii, err)\n        np.add.at(grad, jj, -err)\n        grad += (2.0 * ridge / K) * s\n\n        return loss, grad\n\n    result = minimize(\n        fun=lambda s: loss_grad(s)[0],\n        x0=np.zeros(K, dtype=float),\n        jac=lambda s: loss_grad(s)[1],\n        method="L-BFGS-B",\n        options={"maxiter": 1000, "ftol": 1e-12},\n    )\n\n    score = result.x - np.mean(result.x)\n    pred = expit(score[ii] - score[jj])\n\n    return score, {\n        "success": bool(result.success),\n        "pairwise_mae": float(np.mean(np.abs(pred - target))),\n        "pairwise_rmse": float(np.sqrt(np.mean((pred - target) ** 2))),\n        "objective": float(result.fun),\n    }\n\n\n# ================================================================\n# STEP 4 — Full static VAEP vs interval vs RAR table\n# ================================================================\n\ndef build_static_ranking_table(pm, cohort):\n    player_ids, V, rank_draws = bootstrap_full_cohort(\n        pm,\n        cohort,\n        B=FULL_BOOTSTRAP_B,\n        seed=SEED + 100,\n    )\n\n    P = pairwise_probability_matrix(V)\n    pcs, fit_info = fit_pairwise_consensus(P)\n\n    rar = ranks_desc(pcs)\n    K = len(player_ids)\n\n    q025_rank = np.quantile(rank_draws, 0.025, axis=0)\n    q50_rank = np.quantile(rank_draws, 0.50, axis=0)\n    q975_rank = np.quantile(rank_draws, 0.975, axis=0)\n\n    rows = []\n\n    cohort_ix = cohort.set_index("player_id")\n\n    for i, pid in enumerate(player_ids):\n        p_i = np.delete(P[i, :], i)\n\n        strongly_above = int(\n            np.sum(p_i <= (1.0 - PAIRWISE_SUPPORT))\n        )\n        strongly_below = int(\n            np.sum(p_i >= PAIRWISE_SUPPORT)\n        )\n\n        supported_low = 1 + strongly_above\n        supported_high = K - strongly_below\n\n        resolved = (\n            (p_i >= PAIRWISE_SUPPORT)\n            | (p_i <= (1.0 - PAIRWISE_SUPPORT))\n        )\n\n        c = cohort_ix.loc[pid]\n\n        rows.append({\n            "player_id": pid,\n            "player_name": c["player_name"],\n            "is_goalkeeper": bool(c["is_goalkeeper"]),\n            "matches": int(c["total_matches"]),\n            "minutes": float(c["total_minutes"]),\n            "actions": int(c["total_actions"]),\n            "VAEP90": float(c["vaep90"]),\n            "VAEP_rank": int(c["raw_VAEP_rank"]),\n            "bootstrap_median_rank": float(q50_rank[i]),\n            "bootstrap_rank_low_2.5pct": float(q025_rank[i]),\n            "bootstrap_rank_high_97.5pct": float(q975_rank[i]),\n            "bootstrap_rank_interval_width": float(q975_rank[i] - q025_rank[i]),\n            "supported_rank_low": int(supported_low),\n            "supported_rank_high": int(supported_high),\n            "supported_rank_width": int(supported_high - supported_low),\n            "pairwise_resolution_rate": float(np.mean(resolved)),\n            "PCS": float(pcs[i]),\n            "RAR": int(rar[i]),\n            "RAR_minus_VAEP_rank": int(rar[i] - c["raw_VAEP_rank"]),\n            "absolute_rank_shift": int(abs(rar[i] - c["raw_VAEP_rank"])),\n        })\n\n    table = pd.DataFrame(rows)\n\n    # Raw ordering and RAR ordering are both preserved in the same table.\n    table = table.sort_values(\n        ["VAEP_rank", "RAR"]\n    ).reset_index(drop=True)\n\n    # Overall diagnostics\n    median_interval_width = float(\n        table["bootstrap_rank_interval_width"].median()\n    )\n    median_supported_width = float(\n        table["supported_rank_width"].median()\n    )\n    changed_fraction = float(\n        np.mean(table["VAEP_rank"] != table["RAR"])\n    )\n    changed_count = int(\n        np.sum(table["VAEP_rank"] != table["RAR"])\n    )\n\n    pair_i, pair_j = np.triu_indices(K, k=1)\n    pair_probs = P[pair_i, pair_j]\n\n    supported_pair_fraction = float(\n        np.mean(\n            (pair_probs >= PAIRWISE_SUPPORT)\n            | (pair_probs <= (1.0 - PAIRWISE_SUPPORT))\n        )\n    )\n\n    diagnostics = pd.DataFrame([{\n        "dataset": DATASET_LABEL,\n        "eligible_players": K,\n        "bootstrap_draws": FULL_BOOTSTRAP_B,\n        "median_95pct_rank_interval_width": median_interval_width,\n        "median_supported_rank_band_width": median_supported_width,\n        "supported_pair_fraction_90pct": supported_pair_fraction,\n        "players_reordered_by_RAR": changed_count,\n        "fraction_players_reordered_by_RAR": changed_fraction,\n        "median_absolute_RAR_shift": float(table["absolute_rank_shift"].median()),\n        "max_absolute_RAR_shift": int(table["absolute_rank_shift"].max()),\n        "BT_fit_success": fit_info["success"],\n        "BT_pairwise_mae": fit_info["pairwise_mae"],\n        "BT_pairwise_rmse": fit_info["pairwise_rmse"],\n    }])\n\n    # Pairwise output file.\n    pair_rows = []\n    names = cohort_ix["player_name"].to_dict()\n\n    for i in range(K):\n        for j in range(i + 1, K):\n            p = float(P[i, j])\n            if p >= PAIRWISE_SUPPORT:\n                status = "i_above_j_supported"\n            elif p <= (1.0 - PAIRWISE_SUPPORT):\n                status = "j_above_i_supported"\n            else:\n                status = "unresolved"\n\n            pair_rows.append({\n                "player_i_id": player_ids[i],\n                "player_i_name": names[player_ids[i]],\n                "player_j_id": player_ids[j],\n                "player_j_name": names[player_ids[j]],\n                "P_i_above_j": p,\n                "support_status": status,\n                "BT_pred_P_i_above_j": float(expit(pcs[i] - pcs[j])),\n            })\n\n    pairs = pd.DataFrame(pair_rows)\n\n    return table, diagnostics, pairs\n\n\n# ================================================================\n# STEP 5 — Independent raw VAEP vs RAR reproducibility stress test\n# ================================================================\n\ndef bootstrap_fixed_sample(q, B, rng):\n    q = q.reset_index(drop=True)\n    va = q["vaep"].to_numpy(dtype=float)\n    mi = q["minutes_played"].to_numpy(dtype=float)\n    n = len(q)\n\n    idx = rng.integers(0, n, size=(B, n))\n    total_v = va[idx].sum(axis=1)\n    total_m = mi[idx].sum(axis=1)\n\n    return 90.0 * total_v / np.where(total_m > 0, total_m, np.nan)\n\n\ndef rar_from_draws(draws):\n    P = pairwise_probability_matrix_fast(draws)\n    pcs, fit_info = fit_pairwise_consensus(P)\n    rar = ranks_desc(pcs)\n    return rar, fit_info\n\n\ndef pairwise_probability_matrix_fast(V):\n    """\n    Same calculation as the full matrix, without progress bars.\n    Used inside repeated validation loops.\n    """\n    V = np.asarray(V, dtype=float)\n    K = V.shape[1]\n    P = np.full((K, K), 0.5, dtype=float)\n\n    for i in range(K):\n        xi = V[:, i]\n        for j in range(i + 1, K):\n            d = xi - V[:, j]\n            p = float(\n                np.mean(\n                    np.where(\n                        d > 0,\n                        1.0,\n                        np.where(d == 0, 0.5, 0.0),\n                    )\n                )\n            )\n            P[i, j] = p\n            P[j, i] = 1.0 - p\n\n    return P\n\n\ndef run_independent_reproducibility(pm, cohort):\n    validation_cohort = cohort[\n        cohort["total_matches"] >= VALIDATION_MIN_MATCHES\n    ].copy()\n\n    player_ids = validation_cohort["player_id"].tolist()\n\n    print("\\n" + "=" * 96)\n    print("INDEPENDENT REPRODUCIBILITY STRESS TEST")\n    print("=" * 96)\n    print("Fixed validation cohort size:", len(player_ids))\n    print("Minimum observed matches in fixed cohort:", int(validation_cohort["total_matches"].min()))\n    print("n values:", VALIDATION_N_VALUES)\n    print("repetitions per n:", VALIDATION_REPS)\n    print("inner bootstrap draws per sample:", INNER_BOOTSTRAP_B)\n\n    if len(player_ids) < 50:\n        print(\n            "WARNING: validation cohort has fewer than 50 players. "\n            "The test remains valid but is less of a large-leaderboard stress test."\n        )\n\n    min_m = int(validation_cohort["total_matches"].min())\n    invalid_n = [n for n in VALIDATION_N_VALUES if 2 * n > min_m]\n    if invalid_n:\n        raise RuntimeError(\n            f"Validation n values {invalid_n} are impossible because the fixed cohort "\n            f"has a minimum of only {min_m} matches."\n        )\n\n    grouped = {\n        pid: (\n            q.sort_values(["game_date", "game_id"])\n            .reset_index(drop=True)\n        )\n        for pid, q in pm[pm["player_id"].isin(player_ids)].groupby("player_id")\n    }\n\n    rng_master = np.random.default_rng(SEED + 7000)\n    raw_rows = []\n\n    for n in VALIDATION_N_VALUES:\n        print(f"\\nTesting n={n} matches per independent sample...")\n\n        for rep in tqdm(\n            range(VALIDATION_REPS),\n            desc=f"n={n}",\n            unit="rep",\n        ):\n            K = len(player_ids)\n            raw_A = np.empty(K, dtype=float)\n            raw_B = np.empty(K, dtype=float)\n            draws_A = np.empty((INNER_BOOTSTRAP_B, K), dtype=float)\n            draws_B = np.empty((INNER_BOOTSTRAP_B, K), dtype=float)\n\n            for j, pid in enumerate(player_ids):\n                q = grouped[pid]\n\n                chosen = rng_master.choice(\n                    len(q),\n                    size=2 * n,\n                    replace=False,\n                )\n\n                qA = q.iloc[chosen[:n]].copy()\n                qB = q.iloc[chosen[n:]].copy()\n\n                raw_A[j] = vaep90(qA)\n                raw_B[j] = vaep90(qB)\n\n                rng_A = np.random.default_rng(\n                    SEED + 10_000_000 + n * 100_000 + rep * 1000 + j * 2\n                )\n                rng_B = np.random.default_rng(\n                    SEED + 20_000_000 + n * 100_000 + rep * 1000 + j * 2 + 1\n                )\n\n                draws_A[:, j] = bootstrap_fixed_sample(\n                    qA, INNER_BOOTSTRAP_B, rng_A\n                )\n                draws_B[:, j] = bootstrap_fixed_sample(\n                    qB, INNER_BOOTSTRAP_B, rng_B\n                )\n\n            raw_rank_A = ranks_desc(raw_A)\n            raw_rank_B = ranks_desc(raw_B)\n\n            rar_A, fit_A = rar_from_draws(draws_A)\n            rar_B, fit_B = rar_from_draws(draws_B)\n\n            rho_raw = float(\n                spearmanr(raw_rank_A, raw_rank_B).statistic\n            )\n            rho_rar = float(\n                spearmanr(rar_A, rar_B).statistic\n            )\n\n            raw_rows.append({\n                "n_matches_per_sample": n,\n                "rep": rep,\n                "players": K,\n                "raw_VAEP_spearman": rho_raw,\n                "RAR_spearman": rho_rar,\n                "RAR_minus_raw": rho_rar - rho_raw,\n                "RAR_better": int(rho_rar > rho_raw),\n                "RAR_equal": int(np.isclose(rho_rar, rho_raw)),\n                "RAR_worse": int(rho_rar < rho_raw),\n                "BT_A_success": bool(fit_A["success"]),\n                "BT_B_success": bool(fit_B["success"]),\n            })\n\n    raw = pd.DataFrame(raw_rows)\n\n    summary_rows = []\n\n    for n, q in raw.groupby("n_matches_per_sample"):\n        delta = q["RAR_minus_raw"].to_numpy(dtype=float)\n        nonzero = delta[~np.isclose(delta, 0.0)]\n\n        if len(nonzero) == 0:\n            p_value = 1.0\n        else:\n            try:\n                p_value = float(\n                    wilcoxon(\n                        nonzero,\n                        alternative="two-sided",\n                        zero_method="wilcox",\n                    ).pvalue\n                )\n            except Exception:\n                p_value = np.nan\n\n        summary_rows.append({\n            "n_matches_per_sample": int(n),\n            "players": int(q["players"].iloc[0]),\n            "reps": len(q),\n            "median_raw_VAEP_spearman": float(q["raw_VAEP_spearman"].median()),\n            "median_RAR_spearman": float(q["RAR_spearman"].median()),\n            "median_RAR_minus_raw": float(np.median(delta)),\n            "mean_RAR_minus_raw": float(np.mean(delta)),\n            "q025_RAR_minus_raw": float(np.quantile(delta, 0.025)),\n            "q975_RAR_minus_raw": float(np.quantile(delta, 0.975)),\n            "fraction_RAR_better": float(np.mean(delta > 0)),\n            "fraction_equal": float(np.mean(np.isclose(delta, 0.0))),\n            "fraction_RAR_worse": float(np.mean(delta < 0)),\n            "paired_wilcoxon_p": p_value,\n        })\n\n    summary = pd.DataFrame(summary_rows)\n\n    overall = pd.DataFrame([{\n        "dataset": DATASET_LABEL,\n        "fixed_validation_players": len(player_ids),\n        "validation_minimum_matches_required": VALIDATION_MIN_MATCHES,\n        "tested_n_values": ",".join(str(x) for x in VALIDATION_N_VALUES),\n        "reps_per_n": VALIDATION_REPS,\n        "inner_bootstrap_draws": INNER_BOOTSTRAP_B,\n        "pooled_median_RAR_minus_raw": float(raw["RAR_minus_raw"].median()),\n        "pooled_mean_RAR_minus_raw": float(raw["RAR_minus_raw"].mean()),\n        "pooled_fraction_RAR_better": float(np.mean(raw["RAR_minus_raw"] > 0)),\n        "pooled_fraction_equal": float(np.mean(np.isclose(raw["RAR_minus_raw"], 0.0))),\n        "pooled_fraction_RAR_worse": float(np.mean(raw["RAR_minus_raw"] < 0)),\n        "n_points_RAR_higher_median": int(\n            np.sum(summary["median_RAR_spearman"] > summary["median_raw_VAEP_spearman"])\n        ),\n        "n_points_equal_median": int(\n            np.sum(\n                np.isclose(\n                    summary["median_RAR_spearman"],\n                    summary["median_raw_VAEP_spearman"],\n                )\n            )\n        ),\n        "n_points_RAR_lower_median": int(\n            np.sum(summary["median_RAR_spearman"] < summary["median_raw_VAEP_spearman"])\n        ),\n    }])\n\n    return validation_cohort, raw, summary, overall\n\n\n\n# ================================================================\n# BALLON D\'OR MATCHING AND EXTERNAL AGREEMENT\n# ================================================================\n\ndef normalize_text(x):\n    x = unicodedata.normalize("NFKD", str(x))\n    x = "".join(ch for ch in x if not unicodedata.combining(ch))\n    x = x.lower()\n    x = re.sub(r"[^a-z0-9]+", " ", x)\n    return " ".join(x.split())\n\n\ndef tokens_match(text, token_list):\n    t = set(normalize_text(text).split())\n    return all(normalize_text(tok) in t for tok in token_list)\n\n\ndef team_matches(observed_team, expected_team):\n    a = normalize_text(observed_team)\n    b = normalize_text(expected_team)\n\n    # StatsBomb naming can vary slightly for Atlético.\n    aliases = {\n        "atletico madrid": ["atletico madrid", "atletico de madrid"],\n        "real madrid": ["real madrid"],\n        "barcelona": ["barcelona", "fc barcelona"],\n    }\n\n    acceptable = aliases.get(b, [b])\n    return any(x in a for x in acceptable)\n\n\ndef build_ballondor_cohort(cohort):\n    rows = []\n    audit_rows = []\n\n    for ref in BALLON_DOR:\n        team_candidates = cohort[\n            cohort["team_name"].apply(\n                lambda x: team_matches(x, ref["team_2015_16"])\n            )\n        ].copy()\n\n        matched = []\n        match_method = "alias_tokens"\n\n        for r in team_candidates.itertuples(index=False):\n            for alias_tokens in ref["aliases"]:\n                if tokens_match(r.player_name, alias_tokens):\n                    matched.append(r)\n                    break\n\n        # Conservative fallback: if aliases fail, compare normalized Ballon name\n        # against same-team candidates by token overlap. This is mainly a\n        # diagnostic guard for naming variants, not a general fuzzy join.\n        if len(matched) == 0:\n            ref_tokens = set(normalize_text(ref["ballon_name"]).split())\n            scored = []\n\n            for r in team_candidates.itertuples(index=False):\n                player_tokens = set(normalize_text(r.player_name).split())\n                overlap = len(ref_tokens & player_tokens)\n                scored.append((overlap, str(r.player_name), r))\n\n            scored.sort(key=lambda x: (-x[0], x[1]))\n\n            # Only auto-resolve if there is a unique candidate with at least one\n            # exact normalized token in common. Otherwise stop for manual review.\n            if scored and scored[0][0] >= 1:\n                top_score = scored[0][0]\n                top = [x for x in scored if x[0] == top_score]\n                if len(top) == 1:\n                    matched = [top[0][2]]\n                    match_method = "same_team_token_fallback"\n\n        if len(matched) != 1:\n            audit_rows.append({\n                "ballon_name": ref["ballon_name"],\n                "expected_team": ref["team_2015_16"],\n                "published_rank": ref["published_rank"],\n                "points": ref["points"],\n                "status": "MATCH_FAILED" if len(matched) == 0 else "AMBIGUOUS",\n                "candidate_count": len(matched),\n                "match_method": "",\n                "matched_player_name": "|".join(\n                    sorted({str(x.player_name) for x in matched})\n                ),\n                "same_team_candidate_names": " | ".join(\n                    sorted(team_candidates["player_name"].astype(str).tolist())\n                ),\n            })\n            continue\n\n        m = matched[0]\n        row = {\n            "player_id": m.player_id,\n            "player_name": m.player_name,\n            "team_name": m.team_name,\n            "is_goalkeeper": bool(m.is_goalkeeper),\n            "total_matches": int(m.total_matches),\n            "total_minutes": float(m.total_minutes),\n            "total_actions": int(m.total_actions),\n            "total_vaep": float(m.total_vaep),\n            "offensive_value": float(m.offensive_value),\n            "defensive_value": float(m.defensive_value),\n            "goals": int(m.goals),\n            "goalkeeper_match_rows": int(m.goalkeeper_match_rows),\n            "vaep90": float(m.vaep90),\n            "raw_VAEP_rank_base_league": int(m.raw_VAEP_rank),\n            "ballon_name": ref["ballon_name"],\n            "ballon_published_rank": int(ref["published_rank"]),\n            "ballon_points": int(ref["points"]),\n        }\n        rows.append(row)\n\n        audit_rows.append({\n            "ballon_name": ref["ballon_name"],\n            "expected_team": ref["team_2015_16"],\n            "published_rank": ref["published_rank"],\n            "points": ref["points"],\n            "status": "MATCHED",\n            "candidate_count": 1,\n            "match_method": match_method,\n            "matched_player_name": str(m.player_name),\n            "same_team_candidate_names": "",\n        })\n\n    audit = pd.DataFrame(audit_rows)\n    audit.to_csv(OUT_ROOT / "01_ballondor_name_matching_audit.csv", index=False)\n\n    failures = audit[audit["status"] != "MATCHED"]\n    if len(failures):\n        print("\\nBallon d\'Or matching audit:")\n        print(audit.to_string(index=False))\n\n        candidate_export = cohort[\n            cohort["team_name"].apply(\n                lambda x: any(\n                    team_matches(x, t)\n                    for t in ["Real Madrid", "Barcelona", "Atlético Madrid"]\n                )\n            )\n        ][\n            ["player_id", "player_name", "team_name",\n             "total_matches", "total_minutes", "total_actions", "vaep90"]\n        ].sort_values(["team_name", "player_name"])\n\n        candidate_export.to_csv(\n            OUT_ROOT / "01b_laliga_candidate_names_for_debugging.csv",\n            index=False,\n        )\n\n        raise RuntimeError(\n            "One or more Ballon d\'Or nominees could not be matched uniquely "\n            "to the eligible StatsBomb La Liga cohort. The audit now includes "\n            "the actual same-team candidate names. Inspect "\n            "01_ballondor_name_matching_audit.csv and "\n            "01b_laliga_candidate_names_for_debugging.csv."\n        )\n\n    q = pd.DataFrame(rows)\n\n    if q["player_id"].duplicated().any():\n        raise RuntimeError("A StatsBomb player was matched to more than one Ballon d\'Or nominee.")\n\n    # Restricted external rank among the 13 analysed La Liga nominees.\n    # Average ranks handle official voting ties correctly.\n    q["ballon_restricted_rank"] = (\n        q["ballon_points"]\n        .rank(ascending=False, method="average")\n        .astype(float)\n    )\n\n    q = q.sort_values(\n        ["ballon_restricted_rank", "ballon_name"]\n    ).reset_index(drop=True)\n\n    # Re-rank raw VAEP inside the Ballon cohort only.\n    q["raw_VAEP_rank"] = (\n        q["vaep90"]\n        .rank(ascending=False, method="average")\n        .astype(int)\n    )\n\n    print("\\n" + "=" * 96)\n    print("BALLON D\'OR × STATSBOMB MATCHING")\n    print("=" * 96)\n    print("Matched La Liga nominees:", len(q), "/", len(BALLON_DOR))\n    print(\n        q[\n            [\n                "ballon_name", "ballon_published_rank", "ballon_points",\n                "player_name", "team_name", "total_matches", "vaep90"\n            ]\n        ].to_string(index=False)\n    )\n\n    if len(q) != len(BALLON_DOR):\n        raise RuntimeError(\n            f"Expected {len(BALLON_DOR)} matched Ballon d\'Or nominees, "\n            f"but obtained {len(q)}."\n        )\n\n    return q\n\n\ndef ranking_agreement(ref_rank, pred_rank):\n    ref = np.asarray(ref_rank, dtype=float)\n    pred = np.asarray(pred_rank, dtype=float)\n\n    rho = float(spearmanr(ref, pred).statistic)\n    tau = float(kendalltau(ref, pred, variant="b").statistic)\n    mae = float(np.mean(np.abs(ref - pred)))\n\n    return rho, tau, mae\n\n\ndef pairwise_order_accuracy(ref_rank, pred_rank):\n    ref = np.asarray(ref_rank, dtype=float)\n    pred = np.asarray(pred_rank, dtype=float)\n\n    correct = 0\n    total = 0\n\n    for i in range(len(ref)):\n        for j in range(i + 1, len(ref)):\n            # External ties provide no pairwise ordering target.\n            if np.isclose(ref[i], ref[j]):\n                continue\n\n            ref_dir = np.sign(ref[j] - ref[i])   # + if i ranks better\n            pred_dir = np.sign(pred[j] - pred[i])\n\n            if pred_dir == 0:\n                continue\n\n            total += 1\n            correct += int(ref_dir == pred_dir)\n\n    return float(correct / total) if total else np.nan, int(total)\n\n\ndef topk_overlap(ref_rank, pred_rank, k):\n    ref = np.asarray(ref_rank, dtype=float)\n    pred = np.asarray(pred_rank, dtype=float)\n\n    ref_idx = set(np.argsort(ref)[:k].tolist())\n    pred_idx = set(np.argsort(pred)[:k].tolist())\n\n    return int(len(ref_idx & pred_idx))\n\n\ndef evaluate_external_agreement(static_table):\n    q = static_table.sort_values("ballon_restricted_rank").reset_index(drop=True)\n\n    ref = q["ballon_restricted_rank"].to_numpy(dtype=float)\n    raw = q["VAEP_rank"].to_numpy(dtype=float)\n    rar = q["RAR"].to_numpy(dtype=float)\n\n    raw_rho, raw_tau, raw_mae = ranking_agreement(ref, raw)\n    rar_rho, rar_tau, rar_mae = ranking_agreement(ref, rar)\n\n    raw_pair, n_pairs = pairwise_order_accuracy(ref, raw)\n    rar_pair, _ = pairwise_order_accuracy(ref, rar)\n\n    rows = [{\n        "players": len(q),\n        "ordered_external_pairs": n_pairs,\n        "raw_VAEP_spearman": raw_rho,\n        "RAR_spearman": rar_rho,\n        "RAR_minus_raw_spearman": rar_rho - raw_rho,\n        "raw_VAEP_kendall_tau_b": raw_tau,\n        "RAR_kendall_tau_b": rar_tau,\n        "RAR_minus_raw_kendall": rar_tau - raw_tau,\n        "raw_VAEP_mean_absolute_rank_error": raw_mae,\n        "RAR_mean_absolute_rank_error": rar_mae,\n        "RAR_minus_raw_MAE": rar_mae - raw_mae,\n        "raw_VAEP_pairwise_order_accuracy": raw_pair,\n        "RAR_pairwise_order_accuracy": rar_pair,\n        "RAR_minus_raw_pairwise_accuracy": rar_pair - raw_pair,\n        "raw_top3_overlap": topk_overlap(ref, raw, 3),\n        "RAR_top3_overlap": topk_overlap(ref, rar, 3),\n        "raw_top5_overlap": topk_overlap(ref, raw, 5),\n        "RAR_top5_overlap": topk_overlap(ref, rar, 5),\n        "raw_top6_overlap": topk_overlap(ref, raw, 6),\n        "RAR_top6_overlap": topk_overlap(ref, rar, 6),\n    }]\n\n    return pd.DataFrame(rows)\n\n\ndef annotate_pairwise_external_agreement(pairwise_table, static_table):\n    ext = static_table.set_index("player_id")["ballon_restricted_rank"].to_dict()\n\n    q = pairwise_table.copy()\n\n    def external_relation(row):\n        ri = float(ext[row["player_i_id"]])\n        rj = float(ext[row["player_j_id"]])\n\n        if np.isclose(ri, rj):\n            return "ballon_tie"\n        if ri < rj:\n            return "i_above_j"\n        return "j_above_i"\n\n    q["ballon_relation"] = q.apply(external_relation, axis=1)\n\n    def support_agrees(row):\n        status = row["support_status"]\n        relation = row["ballon_relation"]\n\n        if relation == "ballon_tie" or status == "unresolved":\n            return np.nan\n\n        if status == "i_above_j_supported":\n            return int(relation == "i_above_j")\n\n        if status == "j_above_i_supported":\n            return int(relation == "j_above_i")\n\n        return np.nan\n\n    q["supported_direction_agrees_with_ballondor"] = q.apply(\n        support_agrees,\n        axis=1,\n    )\n\n    return q\n\n\ndef bootstrap_nominee_sample(q, B, rng):\n    q = q.reset_index(drop=True)\n    va = q["vaep"].to_numpy(dtype=float)\n    mi = q["minutes_played"].to_numpy(dtype=float)\n    n = len(q)\n\n    idx = rng.integers(0, n, size=(B, n))\n    total_v = va[idx].sum(axis=1)\n    total_m = mi[idx].sum(axis=1)\n\n    return 90.0 * total_v / np.where(total_m > 0, total_m, np.nan)\n\n\ndef run_external_agreement_stress_test(pm, nominee_cohort):\n    player_ids = nominee_cohort["player_id"].tolist()\n\n    min_m = int(nominee_cohort["total_matches"].min())\n    invalid = [n for n in EXTERNAL_N_VALUES if n > min_m]\n    if invalid:\n        raise RuntimeError(\n            f"External n values {invalid} exceed the minimum observed "\n            f"nominee exposure of {min_m} matches."\n        )\n\n    grouped = {\n        pid: (\n            q.sort_values(["game_date", "game_id"])\n            .reset_index(drop=True)\n        )\n        for pid, q in pm[\n            pm["player_id"].isin(player_ids)\n        ].groupby("player_id")\n    }\n\n    ref_map = nominee_cohort.set_index("player_id")["ballon_restricted_rank"].to_dict()\n    ref = np.array([float(ref_map[pid]) for pid in player_ids], dtype=float)\n\n    rng_master = np.random.default_rng(SEED + 8000)\n    rows = []\n\n    print("\\n" + "=" * 96)\n    print("LIMITED-MATCH EXTERNAL-AGREEMENT STRESS TEST")\n    print("=" * 96)\n    print("Nominees:", len(player_ids))\n    print("Minimum observed nominee matches:", min_m)\n    print("n values:", EXTERNAL_N_VALUES)\n    print("repetitions per n:", EXTERNAL_REPS)\n    print("inner bootstrap draws:", INNER_BOOTSTRAP_B)\n\n    for n in EXTERNAL_N_VALUES:\n        print(f"\\nTesting n={n} league matches per nominee...")\n\n        for rep in tqdm(range(EXTERNAL_REPS), desc=f"external n={n}", unit="rep"):\n            K = len(player_ids)\n            raw_values = np.empty(K, dtype=float)\n            draws = np.empty((INNER_BOOTSTRAP_B, K), dtype=float)\n\n            for j, pid in enumerate(player_ids):\n                q = grouped[pid]\n                chosen = rng_master.choice(len(q), size=n, replace=False)\n                qs = q.iloc[chosen].copy()\n\n                raw_values[j] = vaep90(qs)\n\n                rng_b = np.random.default_rng(\n                    SEED + 30_000_000 + n * 100_000 + rep * 1000 + j\n                )\n                draws[:, j] = bootstrap_nominee_sample(\n                    qs,\n                    INNER_BOOTSTRAP_B,\n                    rng_b,\n                )\n\n            raw_rank = ranks_desc(raw_values)\n            rar, fit_info = rar_from_draws(draws)\n\n            raw_rho, raw_tau, raw_mae = ranking_agreement(ref, raw_rank)\n            rar_rho, rar_tau, rar_mae = ranking_agreement(ref, rar)\n\n            raw_pair, n_pairs = pairwise_order_accuracy(ref, raw_rank)\n            rar_pair, _ = pairwise_order_accuracy(ref, rar)\n\n            rows.append({\n                "n_matches_per_nominee": n,\n                "rep": rep,\n                "players": K,\n                "ordered_external_pairs": n_pairs,\n                "raw_spearman": raw_rho,\n                "RAR_spearman": rar_rho,\n                "delta_spearman": rar_rho - raw_rho,\n                "raw_kendall_tau_b": raw_tau,\n                "RAR_kendall_tau_b": rar_tau,\n                "delta_kendall": rar_tau - raw_tau,\n                "raw_MAE": raw_mae,\n                "RAR_MAE": rar_mae,\n                "delta_MAE": rar_mae - raw_mae,\n                "raw_pairwise_accuracy": raw_pair,\n                "RAR_pairwise_accuracy": rar_pair,\n                "delta_pairwise_accuracy": rar_pair - raw_pair,\n                "BT_success": bool(fit_info["success"]),\n            })\n\n    raw = pd.DataFrame(rows)\n\n    summary_rows = []\n    for n, q in raw.groupby("n_matches_per_nominee"):\n        d_rho = q["delta_spearman"].to_numpy(dtype=float)\n        d_tau = q["delta_kendall"].to_numpy(dtype=float)\n        d_mae = q["delta_MAE"].to_numpy(dtype=float)\n\n        nonzero = d_rho[~np.isclose(d_rho, 0.0)]\n        if len(nonzero):\n            try:\n                p = float(\n                    wilcoxon(\n                        nonzero,\n                        alternative="two-sided",\n                        zero_method="wilcox",\n                    ).pvalue\n                )\n            except Exception:\n                p = np.nan\n        else:\n            p = 1.0\n\n        summary_rows.append({\n            "n_matches_per_nominee": int(n),\n            "players": int(q["players"].iloc[0]),\n            "reps": len(q),\n            "median_raw_spearman": float(q["raw_spearman"].median()),\n            "median_RAR_spearman": float(q["RAR_spearman"].median()),\n            "median_delta_spearman": float(np.median(d_rho)),\n            "mean_delta_spearman": float(np.mean(d_rho)),\n            "fraction_RAR_higher_spearman": float(np.mean(d_rho > 0)),\n            "paired_wilcoxon_p_spearman": p,\n            "median_raw_kendall_tau_b": float(q["raw_kendall_tau_b"].median()),\n            "median_RAR_kendall_tau_b": float(q["RAR_kendall_tau_b"].median()),\n            "median_delta_kendall": float(np.median(d_tau)),\n            "median_raw_MAE": float(q["raw_MAE"].median()),\n            "median_RAR_MAE": float(q["RAR_MAE"].median()),\n            "median_delta_MAE": float(np.median(d_mae)),\n            "fraction_RAR_lower_MAE": float(np.mean(d_mae < 0)),\n            "median_raw_pairwise_accuracy": float(q["raw_pairwise_accuracy"].median()),\n            "median_RAR_pairwise_accuracy": float(q["RAR_pairwise_accuracy"].median()),\n        })\n\n    summary = pd.DataFrame(summary_rows)\n\n    overall = pd.DataFrame([{\n        "players": len(player_ids),\n        "tested_n_values": ",".join(map(str, EXTERNAL_N_VALUES)),\n        "reps_per_n": EXTERNAL_REPS,\n        "inner_bootstrap_draws": INNER_BOOTSTRAP_B,\n        "pooled_median_delta_spearman": float(raw["delta_spearman"].median()),\n        "pooled_mean_delta_spearman": float(raw["delta_spearman"].mean()),\n        "pooled_fraction_RAR_higher_spearman": float(np.mean(raw["delta_spearman"] > 0)),\n        "pooled_median_delta_kendall": float(raw["delta_kendall"].median()),\n        "pooled_median_delta_MAE": float(raw["delta_MAE"].median()),\n        "pooled_fraction_RAR_lower_MAE": float(np.mean(raw["delta_MAE"] < 0)),\n    }])\n\n    return raw, summary, overall\n\n\n# ================================================================\n# RUN COMPLETE ANALYSIS\n# ================================================================\n\npm, league_cohort, model_metrics, build_meta = build_laliga_player_match()\n\nleague_cohort.to_csv(\n    OUT_ROOT / "00_base_eligible_laliga_players.csv",\n    index=False,\n)\n\nnominee_cohort = build_ballondor_cohort(league_cohort)\n\nnominee_cohort.to_csv(\n    OUT_ROOT / "02_ballondor_laliga_nominee_cohort.csv",\n    index=False,\n)\n\n# Full ReliableVAEP / PCS / RAR only among the matched Ballon d\'Or nominees.\nstatic_table, static_diagnostics, pairwise_table = build_static_ranking_table(\n    pm,\n    nominee_cohort,\n)\n\n# Merge official Ballon d\'Or metadata onto the static ranking table.\nmeta_cols = nominee_cohort[\n    [\n        "player_id",\n        "ballon_name",\n        "ballon_published_rank",\n        "ballon_points",\n        "ballon_restricted_rank",\n        "team_name",\n        "raw_VAEP_rank_base_league",\n    ]\n].copy()\n\nstatic_table = static_table.drop(\n    columns=["team_name"],\n    errors="ignore",\n).merge(\n    meta_cols,\n    on="player_id",\n    how="left",\n    validate="one_to_one",\n)\n\nstatic_table["VAEP_abs_error_vs_ballondor"] = (\n    static_table["VAEP_rank"].astype(float)\n    - static_table["ballon_restricted_rank"].astype(float)\n).abs()\n\nstatic_table["RAR_abs_error_vs_ballondor"] = (\n    static_table["RAR"].astype(float)\n    - static_table["ballon_restricted_rank"].astype(float)\n).abs()\n\nstatic_table.to_csv(\n    OUT_ROOT / "03_FULL_ballondor_VAEP_RAR_comparison.csv",\n    index=False,\n)\n\nstatic_table.sort_values(\n    ["ballon_restricted_rank", "ballon_name"]\n).to_csv(\n    OUT_ROOT / "04_ballondor_ordered_comparison.csv",\n    index=False,\n)\n\nstatic_table.sort_values(\n    ["VAEP_rank", "ballon_restricted_rank"]\n).to_csv(\n    OUT_ROOT / "05_VAEP_ordered_nominees.csv",\n    index=False,\n)\n\nstatic_table.sort_values(\n    ["RAR", "ballon_restricted_rank"]\n).to_csv(\n    OUT_ROOT / "06_RAR_ordered_nominees.csv",\n    index=False,\n)\n\nexternal_static = evaluate_external_agreement(static_table)\nexternal_static.to_csv(\n    OUT_ROOT / "07_fullseason_external_agreement_metrics.csv",\n    index=False,\n)\n\npairwise_annotated = annotate_pairwise_external_agreement(\n    pairwise_table,\n    static_table,\n)\npairwise_annotated.to_csv(\n    OUT_ROOT / "08_pairwise_probabilities_with_ballondor_direction.csv",\n    index=False,\n)\n\nstatic_diagnostics.to_csv(\n    OUT_ROOT / "09_static_reliablevaep_diagnostics.csv",\n    index=False,\n)\n\nsupported_external = pairwise_annotated[\n    pairwise_annotated["supported_direction_agrees_with_ballondor"].notna()\n].copy()\n\nif len(supported_external):\n    supported_agreement_rate = float(\n        supported_external["supported_direction_agrees_with_ballondor"].mean()\n    )\nelse:\n    supported_agreement_rate = np.nan\n\nprint("\\n" + "=" * 96)\nprint("FULL-SEASON BALLON D\'OR EXTERNAL AGREEMENT")\nprint("=" * 96)\nprint(external_static.round(4).to_string(index=False))\nprint(\n    "Supported ReliableVAEP pair directions agreeing with Ballon d\'Or:",\n    "NA" if not np.isfinite(supported_agreement_rate)\n    else f"{100*supported_agreement_rate:.1f}%",\n    f"(n={len(supported_external)} supported, externally ordered pairs)",\n)\n\nprint("\\nBallon d\'Or ordered comparison:")\nprint(\n    static_table.sort_values(\n        ["ballon_restricted_rank", "ballon_name"]\n    )[\n        [\n            "ballon_name",\n            "ballon_published_rank",\n            "ballon_points",\n            "ballon_restricted_rank",\n            "team_name",\n            "matches",\n            "VAEP90",\n            "VAEP_rank",\n            "bootstrap_rank_low_2.5pct",\n            "bootstrap_rank_high_97.5pct",\n            "PCS",\n            "RAR",\n            "VAEP_abs_error_vs_ballondor",\n            "RAR_abs_error_vs_ballondor",\n        ]\n    ].round(4).to_string(index=False)\n)\n\n# ------------------------------------------------------------\n# Limited-match external-agreement stress test\n# ------------------------------------------------------------\nexternal_raw, external_summary, external_overall = (\n    run_external_agreement_stress_test(pm, nominee_cohort)\n)\n\nexternal_raw.to_csv(\n    OUT_ROOT / "10_external_agreement_replicates.csv",\n    index=False,\n)\nexternal_summary.to_csv(\n    OUT_ROOT / "11_external_agreement_by_n.csv",\n    index=False,\n)\nexternal_overall.to_csv(\n    OUT_ROOT / "12_external_agreement_overall.csv",\n    index=False,\n)\n\nprint("\\n" + "=" * 96)\nprint("LIMITED-MATCH EXTERNAL AGREEMENT: RAW VAEP VS RAR")\nprint("=" * 96)\nprint(external_summary.round(4).to_string(index=False))\n\nprint("\\nOverall limited-match result:")\nprint(external_overall.round(4).to_string(index=False))\n\n\n# ================================================================\n# Figures\n# ================================================================\n\nq = static_table.sort_values(\n    ["ballon_restricted_rank", "ballon_name"]\n).copy()\n\n# 1. Ballon restricted rank vs VAEP and RAR.\nfig, ax = plt.subplots(figsize=(8.5, 6))\nax.scatter(\n    q["ballon_restricted_rank"],\n    q["VAEP_rank"],\n    label="Raw VAEP",\n)\nax.scatter(\n    q["ballon_restricted_rank"],\n    q["RAR"],\n    marker="x",\n    label="RAR",\n)\nmx = max(\n    q["ballon_restricted_rank"].max(),\n    q["VAEP_rank"].max(),\n    q["RAR"].max(),\n)\nax.plot([1, mx], [1, mx], linestyle="--")\nax.set_xlabel("2016 Ballon d\'Or restricted rank")\nax.set_ylabel("Model rank")\nax.set_title("Ballon d\'Or agreement: raw VAEP vs RAR")\nax.invert_xaxis()\nax.invert_yaxis()\nax.legend()\nplt.tight_layout()\nfig.savefig(\n    OUT_ROOT / "20_ballondor_vs_VAEP_RAR.png",\n    dpi=180,\n)\nplt.close(fig)\n\n# 2. Full-season rank interval comparison.\nfig, ax = plt.subplots(figsize=(10, 6))\nordered = q.sort_values("ballon_restricted_rank").reset_index(drop=True)\nx = np.arange(len(ordered))\nmid = ordered["bootstrap_median_rank"].to_numpy(dtype=float)\nlo = ordered["bootstrap_rank_low_2.5pct"].to_numpy(dtype=float)\nhi = ordered["bootstrap_rank_high_97.5pct"].to_numpy(dtype=float)\nyerr = np.vstack([mid - lo, hi - mid])\n\nax.errorbar(\n    x,\n    mid,\n    yerr=yerr,\n    fmt="o",\n    capsize=3,\n    label="ReliableVAEP 95% bootstrap rank interval",\n)\nax.scatter(\n    x,\n    ordered["ballon_restricted_rank"],\n    marker="x",\n    label="Ballon d\'Or restricted rank",\n)\nax.set_xticks(x)\nax.set_xticklabels(ordered["ballon_name"], rotation=60, ha="right")\nax.set_ylabel("Rank")\nax.set_title("Ballon d\'Or position against ReliableVAEP uncertainty")\nax.invert_yaxis()\nax.legend()\nplt.tight_layout()\nfig.savefig(\n    OUT_ROOT / "21_ballondor_vs_bootstrap_rank_intervals.png",\n    dpi=180,\n)\nplt.close(fig)\n\n# 3. Limited-match external agreement.\nfig, ax = plt.subplots(figsize=(8, 5.5))\nax.plot(\n    external_summary["n_matches_per_nominee"],\n    external_summary["median_raw_spearman"],\n    marker="o",\n    label="Raw VAEP",\n)\nax.plot(\n    external_summary["n_matches_per_nominee"],\n    external_summary["median_RAR_spearman"],\n    marker="o",\n    label="RAR",\n)\nax.set_xlabel("League matches sampled per nominee")\nax.set_ylabel("Median Spearman with Ballon d\'Or ordering")\nax.set_title("External ranking agreement under limited match evidence")\nax.set_ylim(-1, 1)\nax.legend()\nplt.tight_layout()\nfig.savefig(\n    OUT_ROOT / "22_external_spearman_by_n.png",\n    dpi=180,\n)\nplt.close(fig)\n\n# 4. RAR Spearman gain.\nfig, ax = plt.subplots(figsize=(8, 5.5))\nax.plot(\n    external_summary["n_matches_per_nominee"],\n    external_summary["median_delta_spearman"],\n    marker="o",\n)\nax.axhline(0.0, linestyle="--")\nax.set_xlabel("League matches sampled per nominee")\nax.set_ylabel("Median Spearman gain (RAR - raw VAEP)")\nax.set_title("Does RAR improve Ballon d\'Or agreement?")\nplt.tight_layout()\nfig.savefig(\n    OUT_ROOT / "23_external_RAR_gain.png",\n    dpi=180,\n)\nplt.close(fig)\n\n\n# ================================================================\n# Paper-facing summary\n# ================================================================\n\next = external_static.iloc[0]\nov = external_overall.iloc[0]\ndiag = static_diagnostics.iloc[0]\n\nsummary_lines = [\n    "ReliableVAEP × 2016 Ballon d\'Or external-validity experiment",\n    "=" * 78,\n    "",\n    "Primary cohort: 2016 Ballon d\'Or nominees who played in La Liga 2015/16.",\n    f"Matched nominees: {len(nominee_cohort)}",\n    f"StatsBomb La Liga matches analysed: {build_meta[\'matches\']}/"\n    f"{build_meta[\'expected_matches\']} "\n    f"({100*build_meta[\'source_coverage_fraction\']:.1f}% source coverage)",\n    f"Full ReliableVAEP bootstrap draws: {FULL_BOOTSTRAP_B}",\n    "",\n    "Full-season external agreement:",\n    f"Raw VAEP Spearman with Ballon ordering: {ext[\'raw_VAEP_spearman\']:.4f}",\n    f"RAR Spearman with Ballon ordering: {ext[\'RAR_spearman\']:.4f}",\n    f"Difference (RAR - raw): {ext[\'RAR_minus_raw_spearman\']:.4f}",\n    f"Raw VAEP Kendall tau-b: {ext[\'raw_VAEP_kendall_tau_b\']:.4f}",\n    f"RAR Kendall tau-b: {ext[\'RAR_kendall_tau_b\']:.4f}",\n    f"Raw VAEP mean absolute rank error: "\n    f"{ext[\'raw_VAEP_mean_absolute_rank_error\']:.3f}",\n    f"RAR mean absolute rank error: {ext[\'RAR_mean_absolute_rank_error\']:.3f}",\n    f"Raw/RAR top-6 overlap: "\n    f"{int(ext[\'raw_top6_overlap\'])} / {int(ext[\'RAR_top6_overlap\'])}",\n    f"Supported pair directions agreeing with Ballon ordering: "\n    + (\n        "NA"\n        if not np.isfinite(supported_agreement_rate)\n        else f"{100*supported_agreement_rate:.1f}% "\n             f"across {len(supported_external)} supported externally ordered pairs"\n    ),\n    "",\n    "ReliableVAEP uncertainty:",\n    f"Median 95% rank-interval width: "\n    f"{diag[\'median_95pct_rank_interval_width\']:.2f}",\n    f"Supported pair fraction at 90%: "\n    f"{100*diag[\'supported_pair_fraction_90pct\']:.1f}%",\n    f"Players reordered by RAR: {int(diag[\'players_reordered_by_RAR\'])}/"\n    f"{int(diag[\'eligible_players\'])}",\n    "",\n    "Limited-match external-agreement stress test:",\n    f"Tested n values: {ov[\'tested_n_values\']}",\n    f"Pooled median Spearman gain (RAR - raw): "\n    f"{ov[\'pooled_median_delta_spearman\']:.4f}",\n    f"RAR had higher Spearman in "\n    f"{100*ov[\'pooled_fraction_RAR_higher_spearman\']:.1f}% of replicates",\n    f"Pooled median Kendall gain: "\n    f"{ov[\'pooled_median_delta_kendall\']:.4f}",\n    f"Pooled median MAE change (RAR - raw): "\n    f"{ov[\'pooled_median_delta_MAE\']:.4f}",\n    f"RAR had lower rank MAE in "\n    f"{100*ov[\'pooled_fraction_RAR_lower_MAE\']:.1f}% of replicates",\n    "",\n    "Interpretation limits:",\n    "1. Ballon d\'Or is an independent human-voting reference, not objective ground truth.",\n    "2. The 2016 Ballon d\'Or award cycle is not identical to the 2015/16 domestic-league window.",\n    "3. This primary analysis intentionally restricts the comparison to La Liga nominees "\n    "so all players are scored by the same competition-specific VAEP model and data context.",\n    "4. International matches, UEFA competition matches, trophies, narrative factors, "\n    "and performances after the 2015/16 league season are outside this VAEP dataset.",\n    "",\n    "Decision rule:",\n    "Describe RAR as having stronger external agreement only if the full-season and/or "\n    "limited-match metrics consistently improve relative to raw VAEP. If they do not, "\n    "report the disagreement directly; do not treat Ballon d\'Or as a target to optimize.",\n]\n\nsummary_text = "\\n".join(summary_lines)\n(OUT_ROOT / "13_paper_facing_summary.txt").write_text(\n    summary_text,\n    encoding="utf-8",\n)\n\nprint("\\n" + summary_text)\n\nzip_path = shutil.make_archive(\n    str(ROOT / "ReliableVAEP_BALLONDOR2016_LALIGA_RESULTS"),\n    "zip",\n    root_dir=OUT_ROOT,\n)\n\nprint("\\n" + "=" * 96)\nprint("ANALYSIS COMPLETE")\nprint("=" * 96)\nprint("Results folder:", OUT_ROOT)\nprint("Results ZIP:", zip_path)\n'

SCRIPT_PATH.write_text(ANALYSIS_SCRIPT, encoding="utf-8")
compile(ANALYSIS_SCRIPT, str(SCRIPT_PATH), "exec")

print("Analysis script written to:", SCRIPT_PATH)
print("Syntax check: OK")
print("Script size:", len(ANALYSIS_SCRIPT), "bytes")
print("Name-matching fix: Pepe surname-component aliases + audited fallback")


Analysis script written to: /kaggle/working/reliablevaep_ballondor2016_laliga.py
Syntax check: OK
Script size: 70518 bytes
Name-matching fix: Pepe surname-component aliases + audited fallback


In [4]:
# ================================================================
# CELL 4 — RUN THE COMPLETE ANALYSIS
# ================================================================
import os
import subprocess
from pathlib import Path

ENV_PY = Path("/kaggle/working/rv_vaep_env/bin/python")
SCRIPT_PATH = Path("/kaggle/working/reliablevaep_ballondor2016_laliga.py")

if not ENV_PY.exists():
    raise RuntimeError(
        "Run Cell 2 first. The isolated Kaggle VAEP environment does not exist."
    )

if not SCRIPT_PATH.exists():
    raise RuntimeError(
        "Run Cell 3 first. The Ballon d'Or analysis script does not exist."
    )

child_env = os.environ.copy()
child_env["MPLBACKEND"] = "Agg"
child_env.pop("MATPLOTLIBRC", None)
child_env.pop("PYTHONPATH", None)
child_env["PYTHONNOUSERSITE"] = "1"

print("Running ReliableVAEP × 2016 Ballon d'Or analysis...")
print("Analysis Python:", ENV_PY)
print()

process = subprocess.Popen(
    [str(ENV_PY), "-u", str(SCRIPT_PATH)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=child_env,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Analysis exited with code {return_code}. "
        "The exact traceback is shown immediately above."
    )

print("\nBallon d'Or external-validity analysis finished successfully.")


Running ReliableVAEP × 2016 Ballon d'Or analysis...
Analysis Python: /kaggle/working/rv_vaep_env/bin/python

ReliableVAEP external validity: 2016 Ballon d'Or × La Liga 2015/16
Primary comparison cohort: 6 Real Madrid Ballon d'Or nominees
Base eligibility: >=5 matches, >=300 minutes, >=150 actions
Full bootstrap draws: 3500
Pairwise support threshold: 0.9
External-agreement n values: [5, 8, 10]
External repetitions per n: 200
Inner bootstrap draws: 500

IMPORTANT: Ballon d'Or is treated as an independent external ranking, not ground truth.
IMPORTANT: 2016 award scope and 2015/16 domestic-league scope are not identical.

StatsBomb La Liga 2015/16 matches found: 380 (100.0% of 380)

La Liga 2015/16: SPADL: 100%|██████████| 380/380 [10:46<00:00,  1.70s/match]

Canonical VAEP features: 100%|██████████| 380/380 [01:13<00:00,  5.16match/s]
Action rows=757,310, features=568, score rate=0.0110, concede rate=0.0024

Cross-fit model means:
fold                          3.0000
fit_matches         

In [5]:
# ================================================================
# CELL 5 — REVIEW THE MAIN RESULTS
# ================================================================
import csv
import html
from pathlib import Path
from IPython.display import display, HTML

OUT = Path("/kaggle/working/rv_ballondor2016_laliga_outputs")

def read_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def table(title, rows, columns=None, limit=100):
    if not rows:
        print(title, ": no rows")
        return

    rows = rows[:limit]
    if columns is None:
        columns = list(rows[0].keys())

    parts = [
        f"<h3>{html.escape(title)}</h3>",
        '<div style="overflow:auto;max-height:700px">',
        '<table style="border-collapse:collapse;width:100%;font-size:13px">',
        "<thead><tr>",
    ]

    for c in columns:
        parts.append(
            f'<th style="border:1px solid #ddd;padding:5px;position:sticky;'
            f'top:0;background:#f3f4f6">{html.escape(c)}</th>'
        )

    parts.append("</tr></thead><tbody>")

    for r in rows:
        parts.append("<tr>")
        for c in columns:
            parts.append(
                f'<td style="border:1px solid #ddd;padding:5px">'
                f'{html.escape(str(r.get(c, "")))}</td>'
            )
        parts.append("</tr>")

    parts.append("</tbody></table></div>")
    display(HTML("".join(parts)))

comparison = OUT / "04_ballondor_ordered_comparison.csv"
agreement = OUT / "07_fullseason_external_agreement_metrics.csv"
stress = OUT / "11_external_agreement_by_n.csv"
summary = OUT / "13_paper_facing_summary.txt"
audit = OUT / "01_ballondor_name_matching_audit.csv"

if audit.exists():
    table(
        "Ballon d'Or nominee → StatsBomb matching audit",
        read_csv(audit),
    )

if comparison.exists():
    table(
        "Ballon d'Or vs raw VAEP vs RAR",
        read_csv(comparison),
        [
            "ballon_name",
            "ballon_published_rank",
            "ballon_points",
            "ballon_restricted_rank",
            "team_name",
            "matches",
            "VAEP90",
            "VAEP_rank",
            "bootstrap_rank_low_2.5pct",
            "bootstrap_rank_high_97.5pct",
            "PCS",
            "RAR",
            "VAEP_abs_error_vs_ballondor",
            "RAR_abs_error_vs_ballondor",
        ],
    )

if agreement.exists():
    table(
        "Full-season external agreement",
        read_csv(agreement),
    )

if stress.exists():
    table(
        "Limited-match external agreement",
        read_csv(stress),
        [
            "n_matches_per_nominee",
            "players",
            "median_raw_spearman",
            "median_RAR_spearman",
            "median_delta_spearman",
            "fraction_RAR_higher_spearman",
            "paired_wilcoxon_p_spearman",
            "median_raw_kendall_tau_b",
            "median_RAR_kendall_tau_b",
            "median_raw_MAE",
            "median_RAR_MAE",
            "fraction_RAR_lower_MAE",
        ],
    )

if summary.exists():
    print("\n" + "=" * 90)
    print(summary.read_text(encoding="utf-8"))

print("\nResults ZIP:")
print("/kaggle/working/ReliableVAEP_BALLONDOR2016_LALIGA_RESULTS.zip")


ballon_name,expected_team,published_rank,points,status,candidate_count,match_method,matched_player_name,same_team_candidate_names
Cristiano Ronaldo,Real Madrid,1,745,MATCHED,1,alias_tokens,Cristiano Ronaldo dos Santos Aveiro,
Gareth Bale,Real Madrid,6,60,MATCHED,1,alias_tokens,Gareth Frank Bale,
Pepe,Real Madrid,9,8,MATCHED,1,alias_tokens,Kléper Laveran Lima Ferreira,
Luka Modrić,Real Madrid,17,1,MATCHED,1,alias_tokens,Luka Modrić,
Toni Kroos,Real Madrid,17,1,MATCHED,1,alias_tokens,Toni Kroos,
Sergio Ramos,Real Madrid,20,0,MATCHED,1,alias_tokens,Sergio Ramos García,


ballon_name,ballon_published_rank,ballon_points,ballon_restricted_rank,team_name,matches,VAEP90,VAEP_rank,bootstrap_rank_low_2.5pct,bootstrap_rank_high_97.5pct,PCS,RAR,VAEP_abs_error_vs_ballondor,RAR_abs_error_vs_ballondor
Cristiano Ronaldo,1,745,1.0,Real Madrid,36,0.24032598712602332,4,2.0,6.0,-0.8197219169602619,4,3.0,3.0
Gareth Bale,6,60,2.0,Real Madrid,23,0.8644941225082788,1,1.0,1.0,4.772441057487819,1,1.0,1.0
Pepe,9,8,3.0,Real Madrid,21,0.1232201273311506,6,4.0,6.0,-2.5896887284954437,6,3.0,3.0
Luka Modrić,17,1,4.5,Real Madrid,32,0.324690878054869,2,2.0,4.0,0.4090711664604392,2,2.5,2.5
Toni Kroos,17,1,4.5,Real Madrid,32,0.31960241747777135,3,2.0,4.0,0.3783726315817473,3,1.5,1.5
Sergio Ramos,20,0,6.0,Real Madrid,23,0.14782145650894638,5,4.0,6.0,-2.1504742100742993,5,1.0,1.0


players,ordered_external_pairs,raw_VAEP_spearman,RAR_spearman,RAR_minus_raw_spearman,raw_VAEP_kendall_tau_b,RAR_kendall_tau_b,RAR_minus_raw_kendall,raw_VAEP_mean_absolute_rank_error,RAR_mean_absolute_rank_error,RAR_minus_raw_MAE,raw_VAEP_pairwise_order_accuracy,RAR_pairwise_order_accuracy,RAR_minus_raw_pairwise_accuracy,raw_top3_overlap,RAR_top3_overlap,raw_top5_overlap,RAR_top5_overlap,raw_top6_overlap,RAR_top6_overlap
6,14,0.17393131069573453,0.17393131069573453,0.0,0.13801311186847084,0.13801311186847084,0.0,2.0,2.0,0.0,0.5714285714285714,0.5714285714285714,0.0,1,1,4,4,6,6


n_matches_per_nominee,players,median_raw_spearman,median_RAR_spearman,median_delta_spearman,fraction_RAR_higher_spearman,paired_wilcoxon_p_spearman,median_raw_kendall_tau_b,median_RAR_kendall_tau_b,median_raw_MAE,median_RAR_MAE,fraction_RAR_lower_MAE
5,6,0.10145993123917847,0.14494275891311212,0.0,0.1,0.008234616140942452,0.13801311186847084,0.13801311186847084,1.8333333333333333,1.8333333333333333,0.05
8,6,0.34786262139146906,0.28988551782622424,0.0,0.11,0.0001861085320676567,0.2760262237369417,0.13801311186847084,1.6666666666666667,1.6666666666666667,0.065
10,6,0.17393131069573453,0.17393131069573453,0.0,0.13,0.07979239298142764,0.13801311186847084,0.13801311186847084,1.8333333333333333,2.0,0.08



ReliableVAEP × 2016 Ballon d'Or external-validity experiment

Primary cohort: 2016 Ballon d'Or nominees who played in La Liga 2015/16.
Matched nominees: 6
StatsBomb La Liga matches analysed: 380/380 (100.0% source coverage)
Full ReliableVAEP bootstrap draws: 3500

Full-season external agreement:
Raw VAEP Spearman with Ballon ordering: 0.1739
RAR Spearman with Ballon ordering: 0.1739
Difference (RAR - raw): 0.0000
Raw VAEP Kendall tau-b: 0.1380
RAR Kendall tau-b: 0.1380
Raw VAEP mean absolute rank error: 2.000
RAR mean absolute rank error: 2.000
Raw/RAR top-6 overlap: 6 / 6
Supported pair directions agreeing with Ballon ordering: 66.7% across 9 supported externally ordered pairs

ReliableVAEP uncertainty:
Median 95% rank-interval width: 2.00
Supported pair fraction at 90%: 60.0%
Players reordered by RAR: 0/6

Limited-match external-agreement stress test:
Tested n values: 5,8,10
Pooled median Spearman gain (RAR - raw): 0.0000
RAR had higher Spearman in 11.3% of replicates
Pooled median 

## Files to send back after the Kaggle run

The most useful file is the **executed `.ipynb`**, because it contains the
traceback if anything fails and the printed results if it completes.

The notebook also creates:

- `01_ballondor_name_matching_audit.csv`
- `03_FULL_ballondor_VAEP_RAR_comparison.csv`
- `07_fullseason_external_agreement_metrics.csv`
- `08_pairwise_probabilities_with_ballondor_direction.csv`
- `11_external_agreement_by_n.csv`
- `13_paper_facing_summary.txt`
- `ReliableVAEP_BALLONDOR2016_LALIGA_RESULTS.zip`

Do not interpret Ballon d'Or agreement as prediction accuracy or ground-truth
accuracy. It is an **external ranking agreement** experiment.
